In [68]:
from pyspark.sql import SparkSession
import getpass

username = getpass.getuser()
spark =SparkSession. \
        builder. \
        config('spark.ui.port','0'). \
        config('spark.shuffle.useOldFetchProtocol','true'). \
        config('spark.sql.warehouse.dir',f"/user/{username}/warehouse"). \
        enableHiveSupport(). \
        master('yarn'). \
        getOrCreate()

In [34]:
spark.sql("select member_id, count(*) as total from itv026162_lending_club.customers group by member_id order by total desc")

member_id,total
e4c167053d5418230...,5
76b577467eda5bdbc...,4
3f87585a20f702838...,4
ad8e5d384dae17e06...,4
e7d8d16928817ec8f...,3
3ae415acd6bbfaac1...,3
5d52e7773cb0efff3...,3
059d401bb603d9a80...,3
a2356725aa9da92c6...,3
d9ce4046daf599732...,3


In [35]:
spark.sql("select * from itv026162_lending_club.customers where member_id like 'e4c167053d5418230%' ")

member_id,emp_title,emp_length,home_ownership,annual_income,address_state,address_zipcode,address_country,grade,sub_grade,verification_status,total_high_credit_limit,application_type,join_annual_income,verification_status_joint,ingest_date
e4c167053d5418230...,null,6,MORTGAGE,55000.0,IL,604xx,USA,B,B5,Verified,138780.0,Individual,null,null,2023-09-15 17:18:...
e4c167053d5418230...,null,6,MORTGAGE,55000.0,IL,604xx,USA,B,B5,Verified,171165.0,Individual,null,null,2023-09-15 17:18:...
e4c167053d5418230...,null,6,MORTGAGE,55000.0,IL,604xx,USA,B,B5,Verified,110907.0,Individual,null,null,2023-09-15 17:18:...
e4c167053d5418230...,null,6,MORTGAGE,55000.0,IL,604xx,USA,B,B5,Verified,129833.0,Individual,null,null,2023-09-15 17:18:...
e4c167053d5418230...,null,6,MORTGAGE,55000.0,IL,604xx,USA,B,B5,Verified,207300.0,Individual,null,null,2023-09-15 17:18:...


if obserevd in the above run, we can see the all the data is same, except the values in the column *total_high_credit_limit* which is varying, apparently it shouldnt be going like that single customer should have one value.
In this case, we cannot choose one !
In this case, stakeholder has been notified !!

In [36]:
spark.sql("select member_id, count(*) as total from itv026162_lending_club.loans_defaulters_delinq group by member_id order by total desc")

member_id,total
e4c167053d5418230...,3
b3d4eb0c5a9239348...,2
58dde656f747cee8b...,2
c16e1f92ae5ccc785...,2
b5ded5638e54e1692...,2
0b44a10bc749eabb2...,2
17d76ba5141e1c33a...,2
343344c3c65b023af...,2
bbe43331566910d55...,2
a93a44c9d83793451...,2


In [37]:
spark.sql("select * from itv026162_lending_club.loans_defaulters_delinq where member_id like 'e4c167053d5418230%' ")

member_id,delinq_2yrs,delinq_amnt,mths_since_last_delinq
e4c167053d5418230...,3,0.0,0
e4c167053d5418230...,1,0.0,0
e4c167053d5418230...,1,0.0,45


so we look into delinq_2yrs, that is how many times the customer was in defaulter, it has three entries, apparently it should be only one entry, thsi creates the ambiguity

In [38]:
spark.sql("select member_id, count(*) as total from itv026162_lending_club.loans_defaulters_detail_rec_enq group by member_id order by total desc")

member_id,total
e3b0c44298fc1c149...,33
e4c167053d5418230...,5
ad8e5d384dae17e06...,4
3f87585a20f702838...,4
76b577467eda5bdbc...,4
c92062bb371842b3d...,3
066ddaa64bee66dff...,3
d3731bad3e6a82515...,3
53789bea7edc660ed...,3
291ca1b09ef11ca3e...,3


In [39]:
spark.sql("select * from itv026162_lending_club.loans_defaulters_detail_rec_enq where member_id like 'e4c167053d5418230%' ")

member_id,pub_rec,pub_rec_bankruptcies,inq_last_6mths
e4c167053d5418230...,0,0,0
e4c167053d5418230...,0,0,3
e4c167053d5418230...,0,0,2
e4c167053d5418230...,0,0,1
e4c167053d5418230...,0,0,0


In [40]:
bad_data_customer_df = spark.sql("""
                                select member_id from(
                                select member_id, count(*) AS total
                                from itv026162_lending_club.customers
                                group by member_id
                                having total > 1)
                                """)

In [41]:
bad_data_customer_df.count()

3157

In [42]:
bad_data_customer_df

member_id
61be6498c93cadf89...
f99c6e9cfe3a7a2d2...
761b2f1e61999e14e...
5130d0087970e032e...
675151e58a628e87b...
7115ace193e13d8f3...
01b2223757c3b62e7...
a2af7506825a7dcff...
a53e2f488d2d76a30...
fc0e468bff11ac7c3...


In [43]:
bad_data_loans_defaulters_df = spark.sql("""
                                select member_id from(
                                select member_id, count(*) AS total
                                from itv026162_lending_club.loans_defaulters_delinq
                                group by member_id
                                having total > 1)
                                """)

In [44]:
bad_data_loans_defaulters_df.count()

173

In [45]:
bad_data_loans_defaulters_df

member_id
cbede54df344cdb94...
8d2c605c7ad9209cc...
bbe43331566910d55...
6918b9861ba5a4c67...
b5ded5638e54e1692...
a93a44c9d83793451...
6c2b63ff231e520d4...
694c6cb86608015e9...
58dde656f747cee8b...
77db5fdf951dd04b2...


In [46]:
bad_data_loans_defaulters_detail_rec_enq_df = spark.sql("""
                                select member_id from(
                                select member_id, count(*) AS total
                                from itv026162_lending_club.loans_defaulters_detail_rec_enq
                                group by member_id
                                having total > 1)
                                """)

In [47]:
bad_data_loans_defaulters_detail_rec_enq_df.count()

3189

In [48]:
bad_data_loans_defaulters_detail_rec_enq_df

member_id
675151e58a628e87b...
f284044b881f218c0...
a53e2f488d2d76a30...
fc0e468bff11ac7c3...
61be6498c93cadf89...
7115ace193e13d8f3...
eebbd89aa7efc13e7...
01b2223757c3b62e7...
a2af7506825a7dcff...
d4782ddad8591f44d...


In [26]:
bad_data_loans_defaulters_detail_rec_enq_df.repartition(1).write \
.format("csv") \
.option("header", True) \
.mode("overwrite") \
.option("path","/user/itv026162/lendingclub/project/bad/bad_data_loans_deafulters_delinq_detail_rec_enq") \
.save()

In [49]:
bad_customer_data_df1 = bad_data_customer_df.select("member_id") \
.union(bad_data_loans_defaulters_df.select("member_id")) \
.union(bad_data_loans_defaulters_detail_rec_enq_df.select("member_id"))

In [50]:
bad_customer_data_dinal_df = bad_customer_data_df1.distinct()

In [51]:
bad_customer_data_dinal_df.count()

3189

In [52]:
bad_customer_data_dinal_df.repartition(1).write \
.format("csv") \
.option("header", True) \
.mode("overwrite") \
.option("path","/user/itv026162/lendingclub/project/bad/bad_customer_data_final_df") \
.save()

In [53]:
bad_customer_data_dinal_df.createOrReplaceTempView("bad_data_customer")

In [55]:
customers_clean_df = spark.sql("""
          select * from itv026162_lending_club.customers
          where member_id NOT IN (select member_id from bad_data_customer)
        """)

In [56]:
customers_clean_df.count()

2254192

In [57]:
customers_clean_df.write \
.format("parquet") \
.mode("overwrite") \
.option("path","/user/itv026162/lendingclub/project/cleaned_new/customers_parquet") \
.save()

In [58]:
loan_defaulters_delinq_df = spark.sql("""
                                      select * from itv026162_lending_club.loans_defaulters_delinq
                                      where member_id NOT IN (select member_id from bad_data_customer)
                                    """)

In [59]:
loan_defaulters_delinq_df.write \
.format("parquet") \
.mode("overwrite") \
.option("path","/user/itv026162/lendingclub/project/cleaned_new/loan_defaulters_delinq_parquet") \
.save()

In [60]:
l_df = spark.sql("""
                select * from itv026162_lending_club.loans_defaulters_detail_rec_enq
                where member_id NOT IN (select member_id from bad_data_customer)
                """)

In [61]:
l_df.write \
.format("parquet") \
.mode("overwrite") \
.option("path","/user/itv026162/lendingclub/project/cleaned_new/loans_deafulters_delinq_detail_rec_enq_parquet") \
.save()

In [62]:
spark.sql("""
        CREATE EXTERNAL TABLE itv026162_lending_club.customers_new(member_id string,emp_title
        string,emp_length int, home_ownership string,annual_income float,
        address_state string,address_zipcode string,address_country string,
        grade string,sub_grade string,verification_status string,
        total_high_credit_limit float,application_type string,
        join_annual_income float,verification_status_joint string,ingest_date timestamp) stored as parquet
        LOCATION
        '/user/itv026162/lendingclub/project/cleaned_new/customers_parquet'
""")

""


In [63]:
spark.sql("""
        CREATE EXTERNAL TABLE
        itv026162_lending_club.loans_defaulters_delinq_new(member_id
        string,delinq_2yrs integer,delinq_amnt float,mths_since_last_delinq integer)
        stored as parquet
        LOCATION
        '/user/itv026162/lendingclub/project/cleaned_new/loan_defaulters_delinq_parquet'
""")

""


In [65]:
spark.sql("""
        CREATE EXTERNAL TABLE
        itv026162_lending_club.loans_defaulters_detail_rec_enq_new(member_id string,
        pub_rec integer, pub_rec_bankruptcies integer, inq_last_6mths integer) stored
        as parquet
        LOCATION
        '/user/itv026162/lendingclub/project/cleaned_new/loans_deafulters_delinq_detail_rec_enq_parquet'
""")


""


In [69]:
spark.sql("""
          select member_id, count(*) as total
          from itv026162_lending_club.customers_new
          group by member_id order by total desc
""")

member_id,total
e5b195c9c02c592ba...,1
cd0a588079dab28a0...,1
7d11f406a0376e663...,1
3f3b96ab3a0c80fa1...,1
d6d4f17fe8a41ec58...,1
07e0369060069338b...,1
ebda0820db12570e5...,1
f1916df56a386a2c7...,1
3f4b66124fb004c1d...,1
9792ba37806b6c17c...,1


In [67]:
spark.stop()